# 第十四课:Token Sampling

讲义:[14-sampling.md](14-sampling.md)。

In [ ]:
import numpy as np

def softmax(x):
    m = x.max()
    e = np.exp(x - m)
    return e / e.sum()

logits = np.array([1.0, 4.0, 2.0, 0.0])
print('p =', np.round(softmax(logits), 4))
print('greedy argmax:', np.argmax(logits))

r1, r2 = np.random.default_rng(42), np.random.default_rng(42)
a = r1.choice(4, p=softmax(logits)); b = r2.choice(4, p=softmax(logits))
print('同 seed 两次:', a, b, '可复现:', a == b)

In [ ]:
# 大数定律:1000 次采样 vs 理论
samples = [np.random.default_rng(s).choice(4, p=softmax(logits)) for s in range(1000)]
counts = np.bincount(samples, minlength=4)
print('实际:', counts)
print('理论:', np.round(softmax(logits) * 1000))

In [ ]:
def sample_token(logits, rng, temperature=1.0):
    p = softmax(np.asarray(logits) / temperature)
    p = p / p.sum()                     # 浮点兜底:概率和精确归一
    return rng.choice(len(p), p=p)

# 生成循环(假模型:固定 logits)
def fake_forward(ids):
    base = np.array([1.0, 4.0, 2.0, 0.0])
    return base + 0.1 * len(ids) * np.array([1, -1, 0, 0])   # 随长度微移

def generate(forward, prompt, n, rng, T=1.0):
    ids = list(prompt)
    for _ in range(n):
        logits = forward(ids)
        ids.append(sample_token(logits, rng, T))
    return ids

rng = np.random.default_rng(0)
print('生成:', generate(fake_forward, [0], 10, rng, T=1.0))
rng = np.random.default_rng(0)
print('重放:', generate(fake_forward, [0], 10, rng, T=1.0), '← 完全一致')

In [ ]:
# TODO(C2): 10000 次统计,输出「实际频率 vs 理论概率」表和最大偏差
# TODO(D1): 伪码/实现:支持 seed 的接口,每请求独立 RNG
# TODO(E1): 构造复读:假模型若 greedy 会循环,验证采样打破循环